In [ ]:
import keras as ks
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
data = np.loadtxt("iris.csv", delimiter=",", skiprows=1)

## Extraemos todas las filas pero no la columna de las clases
X = data[:, 0:-1]

## Extraemos la columna de las clases como listas
Y = data[:, [-1]]

flowers = ["Setosa", "Versicolor", "Virginica"]

In [ ]:
## Creamos conjutos de entrenamiento y validación con registros aleatorios
X_train, X_val, Y_train, Y_val = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

In [ ]:
model = ks.Sequential([
    ks.layers.Input(shape=(4,)),
    ks.layers.Dense(8, activation=ks.activations.relu),
    ks.layers.Dense(8, activation=ks.activations.relu),
    ks.layers.Dense(3, activation=ks.activations.softmax)
])


In [ ]:
model.compile(
    optimizer = ks.optimizers.Adam(),
    loss = ks.losses.sparse_categorical_crossentropy,
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    x = X_train, 
    y = Y_train,
    validation_data=(X_val, Y_val),
    epochs = 400,
    callbacks=[
        ks.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        ),
        ks.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6
        )
    ]
)

In [ ]:
predict = model.predict(X_train[:15])

# Convertimos las probabilidades a clases
y_pred = np.argmax(predict, axis=1)


print(f"{'Real':<15} {'Predict':<15} {'Setosa':>15} {'Versicolor':>15} {'Virginica':>15}")
print("-" * 75)

for i in range(len(y_pred)):
    real = int(Y_train[i][0])
    pred = y_pred[i]

    print(
        f"{flowers[real]:<15} "
        f"{flowers[pred]:<15} "
        f"{predict[i][0]:>14.2%} "
        f"{predict[i][1]:>14.2%} "
        f"{predict[i][2]:>14.2%}"
    )
